# Bob-corn — Modèle final Kaggle

Notebook de référence pour la soumission Kaggle : features mécanistiques, benchmark multi-modèles, calibration sigmoid, export `id,corrosion_risk`.

**Score public Kaggle (1 % du dataset) : Brier 0.21038**


## Convention target

- mois d'observation de corrosion → `corrosion_risk = 1`
- même mois − 24 mois → `corrosion_risk = 0`
- soumission : une probabilité par ligne de `environment_test.csv`, ordre de `sample_submission.csv`


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    PLOTS_AVAILABLE = True
except Exception:
    PLOTS_AVAILABLE = False

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import SelectKBest, mutual_info_classif

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
DATA_DIR = Path("..") / "data" / "raw"

ENV_TRAIN_PATH = DATA_DIR / "environment_training.csv"
CORROSION_PATH = DATA_DIR / "corrosions_training.csv"
ENV_TEST_PATH = DATA_DIR / "environment_test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

for path in [ENV_TRAIN_PATH, CORROSION_PATH, ENV_TEST_PATH, SAMPLE_SUB_PATH]:
    assert path.exists(), f"Fichier introuvable : {path}"

env_train_raw = pd.read_csv(ENV_TRAIN_PATH)
corrosions_raw = pd.read_csv(CORROSION_PATH)
env_test_raw = pd.read_csv(ENV_TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("environment_training:", env_train_raw.shape)
print("corrosions_training:", corrosions_raw.shape)
print("environment_test:", env_test_raw.shape)
print("sample_submission:", sample_sub.shape)
display(env_train_raw.head())
display(corrosions_raw.head())
display(sample_sub.head())


## Contrôles de cohérence


In [ ]:
def quick_audit(env_train, env_test, corrosions, sample):
    print("Colonnes train/test identiques :", list(env_train.columns) == list(env_test.columns))
    print("IDs sample uniques :", sample["id"].is_unique)
    print("Lignes sample == test :", len(sample) == len(env_test))
    print("Avions train :", env_train["aircraft_id"].nunique())
    print("Avions test :", env_test["aircraft_id"].nunique())
    print("Avions corrosion :", corrosions["aircraft_id"].nunique())
    
    key_cols = ["aircraft_id", "year_month"]
    print("Doublons env_train aircraft/month :", env_train.duplicated(key_cols).sum())
    print("Doublons env_test aircraft/month :", env_test.duplicated(key_cols).sum())
    
    missing = pd.concat(
        [
            env_train.isna().sum().rename("train_missing"),
            env_test.isna().sum().rename("test_missing"),
        ],
        axis=1,
    )
    display(missing[missing.sum(axis=1) > 0])

quick_audit(env_train_raw, env_test_raw, corrosions_raw, sample_sub)

## Construction de la target


In [ ]:
def to_month_period(s):
    return pd.to_datetime(s).dt.to_period("M")


def add_period_columns(df):
    out = df.copy()
    out["month"] = pd.PeriodIndex(out["year_month"], freq="M")
    out["month_start_date"] = pd.to_datetime(out["month_start_date"], errors="coerce")
    return out


def build_reference_targets(corrosions):
    c = corrosions.copy()
    c["observation_date"] = pd.to_datetime(c["observation_date"], errors="coerce")
    c["observation_month"] = c["observation_date"].dt.to_period("M")
    c["healthy_month"] = c["observation_month"] - 24
    c["delivery_month"] = pd.PeriodIndex(
        c["aircraft_delivery_year"].astype(str) + "-" + c["aircraft_delivery_month"].astype(str).str.zfill(2),
        freq="M",
    )

    positives = c[["aircraft_id", "observation_month", "delivery_month", "observation_date"]].rename(
        columns={"observation_month": "month"}
    )
    positives["corrosion_risk"] = 1
    positives["target_kind"] = "observed_corrosion"

    negatives = c[["aircraft_id", "healthy_month", "delivery_month", "observation_date"]].rename(
        columns={"healthy_month": "month"}
    )
    negatives["corrosion_risk"] = 0
    negatives["target_kind"] = "healthy_24m_before"

    targets = pd.concat([positives, negatives], ignore_index=True)
    targets["year_month"] = targets["month"].astype(str)
    return targets


env_train = add_period_columns(env_train_raw)
env_test = add_period_columns(env_test_raw)
targets = build_reference_targets(corrosions_raw)

print("Targets reconstruites :", targets.shape)
display(targets["corrosion_risk"].value_counts(dropna=False).rename("count"))
display(targets.head())

## Feature engineering

Variables dérivées du mécanisme corrosion : mouillage (TOW, déliquescence), chlorures, acides, oxydants, dépôts, cycles humide-sec, doses cumulées au sol, âge avion.


In [ ]:
def month_diff(a, b):
    '''Nombre de mois entre deux Period[M].'''
    return (a.dt.year - b.dt.year) * 12 + (a.dt.month - b.dt.month)


def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -50, 50)))


def safe_divide(num, den, eps=1e-30):
    return num / (den.abs() + eps)


def log10_positive(s, eps=1e-30):
    return np.log10(np.clip(s, eps, None))


def aircraft_rolling(df, col, window, agg="mean", min_periods=1):
    rolled = df[col].groupby(df["aircraft_id"]).rolling(window=window, min_periods=min_periods)
    if agg == "mean":
        out = rolled.mean()
    elif agg == "sum":
        out = rolled.sum()
    elif agg == "std":
        out = rolled.std()
    elif agg == "max":
        out = rolled.max()
    else:
        raise ValueError(f"Unsupported agg: {agg}")
    return out.reset_index(level=0, drop=True)


def aircraft_ewm(df, col, span):
    return df.groupby("aircraft_id")[col].transform(lambda s: s.ewm(span=span, adjust=False, min_periods=1).mean())


def prepare_environment_features(env, delivery_info=None, is_test=False):
    df = add_period_columns(env)
    df = df.sort_values(["aircraft_id", "month"]).reset_index(drop=True)

    month_start = df["month"].dt.to_timestamp()
    next_month = (df["month"] + 1).dt.to_timestamp()
    df["minutes_in_month"] = (next_month - month_start).dt.days * 24 * 60
    df["parking_ratio"] = (df["total_parking_minutes"] / df["minutes_in_month"]).clip(0, 1.5)
    df["parking_days"] = df["total_parking_minutes"] / (24 * 60)

    if delivery_info is not None:
        df = df.merge(delivery_info[["aircraft_id", "delivery_month"]].drop_duplicates(), on="aircraft_id", how="left")
    else:
        df["delivery_month"] = pd.NaT

    first_month = df.groupby("aircraft_id")["month"].transform("min")
    delivery_proxy = df["delivery_month"].where(df["delivery_month"].notna(), first_month)
    df["delivery_month_proxy"] = pd.PeriodIndex(delivery_proxy.astype(str), freq="M")
    df["age_months"] = month_diff(df["month"], df["delivery_month_proxy"]).clip(lower=0)
    df["age_years"] = df["age_months"] / 12
    df["months_since_first_record"] = month_diff(df["month"], first_month).clip(lower=0)

    df["calendar_month"] = df["month"].dt.month
    df["calendar_year"] = df["month"].dt.year
    df["month_sin"] = np.sin(2 * np.pi * df["calendar_month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["calendar_month"] / 12)
    df["temperature_k"] = df["temperature"].fillna(df["metar_temperature_c"] + 273.15)

    priority_cols = [
        "total_parking_minutes",
        "parking_ratio",
        "parking_days",
        "metar_temperature_c",
        "metar_relative_humidity",
        "metar_dew_point_c",
        "metar_hour_precipitation",
        "sea_salt_aerosol_003_05_mixing_ratio",
        "sea_salt_aerosol_05_5_mixing_ratio",
        "sea_salt_aerosol_5_20_mixing_ratio",
        "sulphate_aerosol_mixing_ratio",
        "sulphur_dioxide_mass_mixing_ratio",
        "nitrogen_dioxide_mass_mixing_ratio",
        "carbon_monoxide_mass_mixing_ratio",
        "ozone_mass_mixing_ratio",
        "h2o2",
        "hno3",
        "oh",
        "formaldehyde",
        "hydrophilic_organic_matter_aerosol_mixing_ratio",
        "hydrophobic_organic_matter_aerosol_mixing_ratio",
        "hydrophilic_black_carbon_aerosol_mixing_ratio",
        "hydrophobic_black_carbon_aerosol_mixing_ratio",
        "specific_humidity",
    ]
    priority_cols = [c for c in priority_cols if c in df.columns]

    g = df.groupby("aircraft_id", group_keys=False)

    df["cum_parking_days"] = g["parking_days"].cumsum()
    df["cum_parking_ratio"] = g["parking_ratio"].cumsum()

    sea_salt_cols = [c for c in df.columns if c.startswith("sea_salt_aerosol_")]
    dust_cols = [c for c in df.columns if c.startswith("dust_aerosol_")]
    organic_cols = [c for c in df.columns if "organic_matter" in c]
    black_carbon_cols = [c for c in df.columns if "black_carbon" in c]

    df["salt_total"] = df[sea_salt_cols].sum(axis=1)
    df["dust_total"] = df[dust_cols].sum(axis=1)
    df["organic_total"] = df[organic_cols].sum(axis=1)
    df["black_carbon_total"] = df[black_carbon_cols].sum(axis=1)
    df["carbonaceous_total"] = df["organic_total"] + df["black_carbon_total"]
    df["hydrophilic_carbonaceous"] = (
        df["hydrophilic_organic_matter_aerosol_mixing_ratio"]
        + df["hydrophilic_black_carbon_aerosol_mixing_ratio"]
    )
    df["hydrophobic_carbonaceous"] = (
        df["hydrophobic_organic_matter_aerosol_mixing_ratio"]
        + df["hydrophobic_black_carbon_aerosol_mixing_ratio"]
    )
    df["hydrophilic_fraction"] = safe_divide(df["hydrophilic_carbonaceous"], df["carbonaceous_total"])
    df["nox_total"] = df[["nitrogen_monoxide_mass_mixing_ratio", "nitrogen_dioxide_mass_mixing_ratio"]].sum(axis=1)
    df["voc_total"] = df[["ethane", "c3h8", "isoprene", "formaldehyde"]].sum(axis=1)
    df["sulfur_total"] = df[["sulphate_aerosol_mixing_ratio", "sulphur_dioxide_mass_mixing_ratio"]].sum(axis=1)
    df["nitrate_proxy"] = df["hno3"] + df["organic_nitrates"] + df["nitrogen_dioxide_mass_mixing_ratio"]
    df["acid_gas_proxy"] = df["sulfur_total"] + df["nox_total"] + df["hno3"] + df["organic_nitrates"]
    df["oxidant_proxy"] = df["ozone_mass_mixing_ratio"] + df["h2o2"] + df["oh"]
    df["atmospheric_oxidant_ox"] = df["ozone_mass_mixing_ratio"] + df["nitrogen_dioxide_mass_mixing_ratio"]
    df["secondary_inorganic_aerosol_proxy"] = df["sulphate_aerosol_mixing_ratio"] + df["nitrate_proxy"]
    df["particle_deposit_proxy"] = df["salt_total"] + df["dust_total"] + df["sulfur_total"] + df["carbonaceous_total"]
    df["pollution_total"] = df[["carbon_monoxide_mass_mixing_ratio", "ozone_mass_mixing_ratio", "sulphur_dioxide_mass_mixing_ratio"]].sum(axis=1)

    df["sea_salt_fine"] = df["sea_salt_aerosol_003_05_mixing_ratio"]
    df["sea_salt_mid"] = df["sea_salt_aerosol_05_5_mixing_ratio"]
    df["sea_salt_coarse"] = df["sea_salt_aerosol_5_20_mixing_ratio"]
    df["salt_deposition_stokes_proxy"] = (
        (0.265 ** 2) * df["sea_salt_fine"]
        + (2.75 ** 2) * df["sea_salt_mid"]
        + (12.5 ** 2) * df["sea_salt_coarse"]
    )
    df["salt_deposition_soft_proxy"] = (
        1.0 * df["sea_salt_fine"]
        + 3.0 * df["sea_salt_mid"]
        + 8.0 * df["sea_salt_coarse"]
    )
    df["dust_deposition_stokes_proxy"] = (
        (0.29 ** 2) * df["dust_aerosol_003_055_mixing_ratio"]
        + (0.725 ** 2) * df["dust_aerosol_055_09_mixing_ratio"]
        + (10.45 ** 2) * df["dust_aerosol_09_20_mixing_ratio"]
    )

    df["dew_humidity_gap"] = df["metar_temperature_c"] - df["metar_dew_point_c"]
    df["near_dew_point"] = np.clip((3 - df["dew_humidity_gap"]) / 3, 0, 1)
    df["condensation_sigmoid"] = sigmoid((2.0 - df["dew_humidity_gap"]) / 1.5)
    df["rh_over_60"] = np.clip((df["metar_relative_humidity"] - 60) / 40, 0, 1)
    df["rh_over_70"] = np.clip((df["metar_relative_humidity"] - 70) / 30, 0, 1)
    df["rh_over_80"] = np.clip((df["metar_relative_humidity"] - 80) / 20, 0, 1)
    df["iso_tow_soft"] = sigmoid((df["metar_relative_humidity"] - 80) / 4) * sigmoid((df["metar_temperature_c"] - 0) / 2)
    df["nacl_deliquescence"] = sigmoid((df["metar_relative_humidity"] - 75) / 3)
    df["nitrate_deliquescence"] = sigmoid((df["metar_relative_humidity"] - 60) / 5)
    df["sulfate_deliquescence"] = sigmoid((df["metar_relative_humidity"] - 80) / 4)
    df["mixed_salt_deliquescence"] = 1 - (
        (1 - df["nacl_deliquescence"])
        * (1 - df["nitrate_deliquescence"])
        * (1 - df["sulfate_deliquescence"])
    )
    df["low_visibility_fog_proxy"] = sigmoid((3.0 - df["metar_visibility_mi"]) / 0.8) * df["rh_over_80"]
    df["rain_flag"] = (df["metar_hour_precipitation"] > 0).astype(float)
    df["wetness_index"] = (
        0.28 * df["rh_over_80"]
        + 0.18 * df["condensation_sigmoid"]
        + 0.18 * df["mixed_salt_deliquescence"]
        + 0.16 * df["iso_tow_soft"]
        + 0.10 * df["rain_flag"]
        + 0.06 * df["specific_humidity"].rank(pct=True)
        + 0.04 * df["low_visibility_fog_proxy"]
    ).clip(0, 1)
    df["time_of_wetness_days"] = df["parking_days"] * df["wetness_index"]
    df["iso_time_of_wetness_days"] = df["parking_days"] * df["iso_tow_soft"]
    df["condensation_wet_days"] = df["parking_days"] * df["condensation_sigmoid"]
    df["deliquescent_wet_days"] = df["parking_days"] * df["mixed_salt_deliquescence"]
    df["fog_salt_activation"] = df["low_visibility_fog_proxy"] * df["salt_deposition_soft_proxy"] * df["parking_days"]

    df["temp_q10_accel"] = np.power(2.0, (df["metar_temperature_c"] - 20) / 10).clip(0.25, 8)
    R = 8.314
    T_REF = 293.15
    for ea_kj in [20, 40, 60]:
        ea = ea_kj * 1000
        df[f"arrhenius_ea{ea_kj}_accel"] = np.exp((-ea / R) * ((1 / df["temperature_k"]) - (1 / T_REF))).clip(0.05, 20)
    df["thermal_wetness_index"] = df["wetness_index"] * df["temp_q10_accel"]
    df["arrhenius_wetness_ea40"] = df["wetness_index"] * df["arrhenius_ea40_accel"]

    df["chloride_wet_exposure"] = df["salt_total"] * df["time_of_wetness_days"]
    df["chloride_deposition_wet_exposure"] = df["salt_deposition_soft_proxy"] * df["time_of_wetness_days"]
    df["chloride_stokes_wet_exposure"] = df["salt_deposition_stokes_proxy"] * df["time_of_wetness_days"]
    df["acid_wet_exposure"] = df["acid_gas_proxy"] * df["time_of_wetness_days"]
    df["oxidant_wet_exposure"] = df["oxidant_proxy"] * df["time_of_wetness_days"]
    df["deposit_wet_exposure"] = df["particle_deposit_proxy"] * df["time_of_wetness_days"]
    df["salt_acid_synergy"] = df["salt_total"] * df["acid_gas_proxy"] * df["wetness_index"]
    df["salt_oxidant_synergy"] = df["salt_total"] * df["oxidant_proxy"] * df["wetness_index"]
    df["salt_nitrate_synergy"] = df["salt_total"] * df["nitrate_proxy"] * df["mixed_salt_deliquescence"]
    df["salt_sulfate_synergy"] = df["salt_total"] * df["sulfur_total"] * df["sulfate_deliquescence"]
    df["secondary_sulfate_formation_proxy"] = df["sulphur_dioxide_mass_mixing_ratio"] * df["h2o2"] * df["wetness_index"]
    df["secondary_nitrate_formation_proxy"] = df["nitrogen_dioxide_mass_mixing_ratio"] * df["ozone_mass_mixing_ratio"] * df["wetness_index"]
    df["sea_salt_chlorine_activation_proxy"] = df["salt_total"] * df["nitrogen_dioxide_mass_mixing_ratio"] * df["ozone_mass_mixing_ratio"] * df["wetness_index"]
    df["electrolyte_conductivity_proxy"] = df["wetness_index"] * (
        df["salt_deposition_soft_proxy"] + df["secondary_inorganic_aerosol_proxy"] + df["acid_gas_proxy"]
    )
    df["hygroscopic_water_retention"] = df["wetness_index"] * (
        df["salt_total"] + df["secondary_inorganic_aerosol_proxy"] + df["hydrophilic_carbonaceous"]
    )
    df["conductive_soot_cell_proxy"] = df["black_carbon_total"] * df["wetness_index"] * df["salt_total"]
    df["stagnant_wet_deposit"] = df["deposit_wet_exposure"] / (1 + df["metar_wind_speed_kn"])
    df["wind_salt_impingement"] = df["metar_wind_speed_kn"] * df["salt_deposition_soft_proxy"]
    df["drying_power"] = df["metar_wind_speed_kn"] * (1 - df["wetness_index"])
    df["light_rain_activation"] = df["salt_deposition_soft_proxy"] * df["wetness_index"] / (1 + 50 * df["metar_hour_precipitation"])
    df["rain_washoff_proxy"] = df["metar_hour_precipitation"] * df["metar_wind_speed_kn"] * (df["salt_deposition_soft_proxy"] + df["dust_deposition_stokes_proxy"])
    df["net_salt_residence_proxy"] = df["salt_deposition_soft_proxy"] * (1 + df["mixed_salt_deliquescence"]) / (1 + df["rain_washoff_proxy"])
    df["photochemical_smog_proxy"] = df["oxidant_proxy"] * (df["nox_total"] + df["voc_total"]) * df["arrhenius_ea20_accel"]
    df["hot_humid_chloride"] = df["temp_q10_accel"] * df["rh_over_80"] * df["salt_total"]
    df["arrhenius_chloride_attack"] = df["arrhenius_ea40_accel"] * df["chloride_deposition_wet_exposure"]
    df["arrhenius_acid_attack"] = df["arrhenius_ea40_accel"] * df["acid_wet_exposure"]
    df["pitting_nucleation_proxy"] = sigmoid(
        2.5 * df["nacl_deliquescence"]
        + 2.0 * df["condensation_sigmoid"]
        + 1.5 * df["salt_deposition_soft_proxy"].rank(pct=True)
        + 1.0 * df["acid_gas_proxy"].rank(pct=True)
        - 3.5
    )
    df["filiform_undercoating_proxy"] = df["age_years"] * df["nacl_deliquescence"] * df["condensation_sigmoid"] * df["salt_deposition_soft_proxy"]
    df["crevice_differential_aeration_proxy"] = df["stagnant_wet_deposit"] * df["condensation_sigmoid"] * (1 + df["dust_deposition_stokes_proxy"].rank(pct=True))
    df["aged_wet_chloride"] = df["age_years"] * df["chloride_wet_exposure"]
    df["aged_acid_wet"] = df["age_years"] * df["acid_wet_exposure"]
    df["parking_wet_pollution"] = df["parking_ratio"] * df["wetness_index"] * df["pollution_total"]

    for col in priority_cols:
        shifted = df[col]
        for window in [3, 6, 12, 24]:
            df[f"{col}_roll{window}_mean"] = aircraft_rolling(df, col, window, "mean")
        df[f"{col}_cummean"] = g[col].expanding(min_periods=1).mean().reset_index(level=0, drop=True)

        if col not in ["parking_ratio", "parking_days", "total_parking_minutes"]:
            weighted = df[col] * df["parking_days"]
            df[f"{col}_cum_ground_exposure"] = weighted.groupby(df["aircraft_id"]).cumsum()

    dynamic_cols = [
        "metar_relative_humidity",
        "metar_temperature_c",
        "dew_humidity_gap",
        "wetness_index",
        "nacl_deliquescence",
        "mixed_salt_deliquescence",
        "salt_total",
        "salt_deposition_soft_proxy",
        "acid_gas_proxy",
        "oxidant_proxy",
        "particle_deposit_proxy",
        "time_of_wetness_days",
        "electrolyte_conductivity_proxy",
    ]
    for col in dynamic_cols:
        df[f"{col}_lag1"] = g[col].shift(1)
        df[f"{col}_lag3"] = g[col].shift(3)
        df[f"{col}_delta1"] = df[col] - df[f"{col}_lag1"]
        df[f"{col}_roll6_std"] = aircraft_rolling(df, col, 6, "std", min_periods=2)
        df[f"{col}_roll12_std"] = aircraft_rolling(df, col, 12, "std", min_periods=2)
        df[f"{col}_ewm6"] = aircraft_ewm(df, col, 6)
        df[f"{col}_ewm12"] = aircraft_ewm(df, col, 12)
        df[f"{col}_ewm24"] = aircraft_ewm(df, col, 24)

    df["wet_dry_cycle_index"] = df["wetness_index_delta1"].abs() * df["parking_ratio"]
    df["rh_cross_80_cycle"] = (
        ((df["metar_relative_humidity"] >= 80) & (df["metar_relative_humidity_lag1"] < 80))
        | ((df["metar_relative_humidity"] < 80) & (df["metar_relative_humidity_lag1"] >= 80))
    ).astype(float)
    df["salt_concentration_cycle"] = df["salt_total"] * df["wet_dry_cycle_index"]

    dose_cols = [
        "time_of_wetness_days",
        "iso_time_of_wetness_days",
        "condensation_wet_days",
        "deliquescent_wet_days",
        "thermal_wetness_index",
        "arrhenius_wetness_ea40",
        "chloride_wet_exposure",
        "chloride_deposition_wet_exposure",
        "chloride_stokes_wet_exposure",
        "acid_wet_exposure",
        "oxidant_wet_exposure",
        "deposit_wet_exposure",
        "salt_acid_synergy",
        "salt_oxidant_synergy",
        "salt_nitrate_synergy",
        "salt_sulfate_synergy",
        "secondary_sulfate_formation_proxy",
        "secondary_nitrate_formation_proxy",
        "sea_salt_chlorine_activation_proxy",
        "electrolyte_conductivity_proxy",
        "hygroscopic_water_retention",
        "conductive_soot_cell_proxy",
        "stagnant_wet_deposit",
        "wind_salt_impingement",
        "light_rain_activation",
        "net_salt_residence_proxy",
        "photochemical_smog_proxy",
        "arrhenius_chloride_attack",
        "arrhenius_acid_attack",
        "pitting_nucleation_proxy",
        "filiform_undercoating_proxy",
        "crevice_differential_aeration_proxy",
        "hot_humid_chloride",
        "aged_wet_chloride",
        "aged_acid_wet",
        "wet_dry_cycle_index",
        "salt_concentration_cycle",
    ]
    for col in dose_cols:
        df[f"{col}_cum"] = g[col].cumsum()
        df[f"{col}_roll12_sum"] = aircraft_rolling(df, col, 12, "sum")
        df[f"{col}_roll24_sum"] = aircraft_rolling(df, col, 24, "sum")
        df[f"{col}_roll36_sum"] = aircraft_rolling(df, col, 36, "sum")
        df[f"{col}_prev24_sum"] = df.groupby("aircraft_id")[f"{col}_roll24_sum"].shift(24)
        df[f"{col}_recent_vs_prev24"] = (
            (df[f"{col}_roll24_sum"] - df[f"{col}_prev24_sum"])
            / (df[f"{col}_roll24_sum"].abs() + df[f"{col}_prev24_sum"].abs() + 1e-30)
        )
        df[f"{col}_cum_per_year"] = df[f"{col}_cum"] / (df["age_years"] + 0.25)

    df["humid_hot"] = df["metar_relative_humidity"] * df["metar_temperature_c"]
    df["ground_age_interaction"] = df["cum_parking_days"] * df["age_years"]
    df["wet_ground_age_interaction"] = df["time_of_wetness_days_cum"] * df["age_years"]

    log_candidate_cols = [
        c for c in df.columns
        if any(token in c for token in [
            "mixing_ratio", "ethane", "c3h8", "isoprene", "h2o2", "hno3", "oh",
            "formaldehyde", "nitrates", "salt_", "dust_", "acid_", "oxidant_",
            "deposit_", "sulfur_", "nitrate_", "chloride_", "electrolyte_", "hygroscopic_",
            "photochemical_", "pitting_", "filiform_", "crevice_"
        ])
        and pd.api.types.is_numeric_dtype(df[c])
        and not c.startswith("log10_")
    ]
    for col in log_candidate_cols[:180]:
        if (df[col].dropna() >= 0).all():
            df[f"log10_{col}"] = log10_positive(df[col])

    return df


delivery_info = targets[["aircraft_id", "delivery_month"]].drop_duplicates()
train_features_all = prepare_environment_features(env_train_raw, delivery_info=delivery_info)
test_features_all = prepare_environment_features(env_test_raw, delivery_info=None, is_test=True)

print(train_features_all.shape, test_features_all.shape)
display(train_features_all.head(3))

## Familles de features

`wetness`, `chloride`, `acid`, `oxidant`, `deposition`, `cycling`, `aging_ground` — sélection mécanistique finale sans raccourcis calendrier absolu.


## Jeu supervisé


In [ ]:
feature_keys = ["aircraft_id", "month", "year_month"]
train_labeled = targets.merge(
    train_features_all,
    on=["aircraft_id", "month", "year_month"],
    how="left",
    indicator=True,
)

print(train_labeled["_merge"].value_counts())
missing_target_rows = train_labeled[train_labeled["_merge"] != "both"][[
    "aircraft_id", "year_month", "corrosion_risk", "target_kind", "observation_date"
]]
display(missing_target_rows.head(20))

train_labeled = train_labeled[train_labeled["_merge"] == "both"].drop(columns=["_merge"])

else:
    train_labeled["delivery_month_model"] = pd.PeriodIndex(train_labeled["delivery_month_proxy"].astype(str), freq="M")

print("Training rows utilisables :", train_labeled.shape)
display(train_labeled[["aircraft_id", "year_month", "corrosion_risk", "target_kind", "age_months", "cum_parking_days"]].head())

In [ ]:
FEATURE_FAMILIES = {
    "wetness_electrolyte": ["wetness", "humidity", "dew", "rain", "specific_humidity", "time_of_wetness", "condensation", "fog", "electrolyte"],
    "deliquescence": ["deliquescence", "nacl", "nitrate_deliquescence", "sulfate_deliquescence", "mixed_salt"],
    "chloride_salt": ["salt", "chloride", "sea_salt"],
    "acid_pollution": ["acid", "sulph", "sulfur", "nox", "nitrogen", "hno3", "nitrate", "pollution", "secondary_sulfate", "secondary_nitrate"],
    "oxidants": ["oxidant", "ozone", "h2o2", "oh", "photochemical", "ox"],
    "deposition_residence": ["deposit", "dust", "carbon", "organic", "particle", "stokes", "residence", "washoff", "impingement"],
    "cycling_hysteresis": ["cycle", "delta", "roll6_std", "roll12_std", "concentration", "recent_vs_prev"],
    "aging_ground": ["age", "parking", "ground", "cum"],
    "thermal_kinetics": ["temp", "thermal", "hot", "q10", "arrhenius"],
    "localized_corrosion": ["pitting", "filiform", "crevice", "soot_cell", "differential"],
}


def feature_family(feature_name):
    f = feature_name.lower()
    hits = [family for family, keys in FEATURE_FAMILIES.items() if any(k in f for k in keys)]
    return hits[0] if hits else "other"


numeric_preview_cols = [
    c for c in train_labeled.columns
    if c not in {"corrosion_risk"} and pd.api.types.is_numeric_dtype(train_labeled[c])
]

family_rows = []
for c in numeric_preview_cols:
    pos = train_labeled.loc[train_labeled["corrosion_risk"] == 1, c].mean()
    neg = train_labeled.loc[train_labeled["corrosion_risk"] == 0, c].mean()
    pooled_std = train_labeled[c].std()
    if pd.notna(pooled_std) and pooled_std > 0:
        family_rows.append({
            "feature": c,
            "family": feature_family(c),
            "mean_positive": pos,
            "mean_negative": neg,
            "std_diff_positive_minus_negative": (pos - neg) / pooled_std,
        })

mechanism_signal = (
    pd.DataFrame(family_rows)
    .assign(abs_signal=lambda d: d["std_diff_positive_minus_negative"].abs())
    .sort_values("abs_signal", ascending=False)
)

display(mechanism_signal.head(30))
display(
    mechanism_signal
    .groupby("family")["abs_signal"]
    .agg(["count", "mean", "max"])
    .sort_values("max", ascending=False)
)

In [ ]:
if PLOTS_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.countplot(data=train_labeled, x="corrosion_risk", ax=axes[0])
    axes[0].set_title("Balance target reconstruite")
    sns.boxplot(data=train_labeled, x="corrosion_risk", y="age_months", ax=axes[1])
    axes[1].set_title("Âge avion aux points de référence")
    plt.tight_layout()
    plt.show()

display(train_labeled.groupby(["target_kind", "corrosion_risk"]).size().rename("rows"))
display(train_labeled.groupby("corrosion_risk")[["age_months", "cum_parking_days", "parking_ratio"]].mean())

## Modélisation

GroupKFold par `aircraft_id`, métrique Brier. Benchmark sur un jeu de features mécanistiques unique ; le meilleur modèle OOF est calibré (sigmoid) puis réentraîné sur tout le train.


In [ ]:
ID_COLS = {
    "aircraft_id", "year_month", "month_start_date", "month", "delivery_month", "delivery_month_x", "delivery_month_y", "delivery_month_model", "delivery_month_proxy",
    "observation_date", "target_kind", "corrosion_risk"
}

feature_cols = [
    c for c in train_labeled.columns
    if c not in ID_COLS and pd.api.types.is_numeric_dtype(train_labeled[c])
]

non_constant_features = []
for c in feature_cols:
    s = train_labeled[c]
    if s.notna().mean() > 0.80 and s.nunique(dropna=True) > 1:
        non_constant_features.append(c)
all_feature_cols = non_constant_features

MECHANISM_FAMILIES = {
    "wetness_electrolyte", "deliquescence", "chloride_salt", "acid_pollution",
    "oxidants", "deposition_residence", "cycling_hysteresis", "thermal_kinetics",
    "localized_corrosion", "aging_ground",
}
EXCLUDED_SHORTCUT_TERMS = [
    "calendar_year",
    "months_since_first_record",
]

feature_cols = [
    c for c in all_feature_cols
    if feature_family(c) in MECHANISM_FAMILIES
    and not any(term in c for term in EXCLUDED_SHORTCUT_TERMS)
]

X = train_labeled[feature_cols].replace([np.inf, -np.inf], np.nan)
y = train_labeled["corrosion_risk"].astype(int)
groups = train_labeled["aircraft_id"]

print("Nombre de features toutes candidates :", len(all_feature_cols))
print("Nombre de features mécanistiques finales :", len(feature_cols))
print("Nombre d'exemples :", len(X))
display(
    pd.Series([feature_family(c) for c in feature_cols], name="family")
    .value_counts()
    .rename_axis("feature_family")
    .reset_index(name="n_features")
)
display(pd.Series(feature_cols).head(30).to_frame("features"))

In [ ]:
optional_model_packages = {}
for package_name in ["catboost", "xgboost", "lightgbm"]:
    try:
        __import__(package_name)
        optional_model_packages[package_name] = "available - included in benchmark"
    except Exception:
        optional_model_packages[package_name] = "not installed - skipped"

display(pd.Series(optional_model_packages, name="status").to_frame())

In [ ]:
def build_models(n_features=None):
    n_features = len(feature_cols) if n_features is None else n_features
    k_best = min(250, n_features)

    def mi_scores(X_arr, y_arr):
        return mutual_info_classif(X_arr, y_arr, random_state=RANDOM_STATE)

    logistic_mi = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("select", SelectKBest(score_func=mi_scores, k=k_best)),
            ("scaler", RobustScaler()),
            ("model", LogisticRegression(
                max_iter=4000,
                C=0.6,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]
    )

    random_forest_mi = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("select", SelectKBest(score_func=mi_scores, k=k_best)),
            ("model", RandomForestClassifier(
                n_estimators=600,
                min_samples_leaf=8,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]
    )

    extra_trees_mi = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("select", SelectKBest(score_func=mi_scores, k=k_best)),
            ("model", ExtraTreesClassifier(
                n_estimators=700,
                min_samples_leaf=6,
                max_features="sqrt",
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]
    )

    hgb_mi = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("select", SelectKBest(score_func=mi_scores, k=k_best)),
            ("model", HistGradientBoostingClassifier(
                learning_rate=0.03,
                max_iter=500,
                max_leaf_nodes=10,
                min_samples_leaf=18,
                l2_regularization=0.8,
                random_state=RANDOM_STATE,
            )),
        ]
    )

    hgb_smooth_mi = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("select", SelectKBest(score_func=mi_scores, k=k_best)),
            ("model", HistGradientBoostingClassifier(
                learning_rate=0.025,
                max_iter=450,
                max_leaf_nodes=8,
                min_samples_leaf=24,
                l2_regularization=1.2,
                random_state=RANDOM_STATE,
            )),
        ]
    )

    models = {
        "logistic_mi250": logistic_mi,
        "random_forest_mi250": random_forest_mi,
        "extra_trees_mi250": extra_trees_mi,
        "hgb_mi250": hgb_mi,
        "hgb_smooth_mi250": hgb_smooth_mi,
    }

    try:
        from xgboost import XGBClassifier
        models["xgboost_mi250"] = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("select", SelectKBest(score_func=mi_scores, k=k_best)),
                ("model", XGBClassifier(
                    n_estimators=600,
                    learning_rate=0.025,
                    max_depth=3,
                    min_child_weight=6,
                    subsample=0.85,
                    colsample_bytree=0.75,
                    reg_alpha=0.4,
                    reg_lambda=6.0,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    tree_method="hist",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ]
        )
    except Exception:
        pass

    try:
        from lightgbm import LGBMClassifier
        models["lightgbm_mi250"] = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("select", SelectKBest(score_func=mi_scores, k=k_best)),
                ("model", LGBMClassifier(
                    n_estimators=650,
                    learning_rate=0.025,
                    num_leaves=12,
                    min_child_samples=24,
                    subsample=0.85,
                    colsample_bytree=0.75,
                    reg_alpha=0.4,
                    reg_lambda=4.0,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                )),
            ]
        )
    except Exception:
        pass

    try:
        from catboost import CatBoostClassifier
        models["catboost_mi250"] = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("select", SelectKBest(score_func=mi_scores, k=k_best)),
                ("model", CatBoostClassifier(
                    iterations=650,
                    learning_rate=0.025,
                    depth=4,
                    l2_leaf_reg=10,
                    loss_function="Logloss",
                    eval_metric="Logloss",
                    random_seed=RANDOM_STATE,
                    verbose=False,
                )),
            ]
        )
    except Exception:
        pass

    return models


def evaluate_group_cv(models, X, y, groups, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    oof = pd.DataFrame(index=X.index)

    for name, model in models.items():
        pred = np.zeros(len(X), dtype=float)
        fold_scores = []

        for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), start=1):
            m = clone(model)
            m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            p = m.predict_proba(X.iloc[va_idx])[:, 1]
            p = np.clip(p, 0.001, 0.999)
            pred[va_idx] = p
            fold_scores.append(brier_score_loss(y.iloc[va_idx], p))

        oof[name] = pred
        rows.append({
            "model": name,
            "brier_oof": brier_score_loss(y, pred),
            "brier_fold_mean": np.mean(fold_scores),
            "brier_fold_std": np.std(fold_scores),
            "log_loss_oof": log_loss(y, np.clip(pred, 0.001, 0.999)),
            "auc_oof": roc_auc_score(y, pred),
        })

    results = pd.DataFrame(rows).sort_values("brier_oof")
    return results, oof


models = build_models()
cv_results, oof_predictions = evaluate_group_cv(models, X, y, groups, n_splits=5)
display(cv_results)

FINAL_MODEL_NAME = cv_results.iloc[0]["model"]
print("Modèle final choisi sur le même dataset/features :", FINAL_MODEL_NAME)

## Robustesse (stress tests)


In [ ]:
def evaluate_single_split(model, X_data, y_data, train_mask, valid_mask, label):
    train_mask = np.asarray(train_mask)
    valid_mask = np.asarray(valid_mask)
    if valid_mask.sum() < 20 or y_data[valid_mask].nunique() < 2 or y_data[train_mask].nunique() < 2:
        return {"split": label, "brier": np.nan, "log_loss": np.nan, "auc": np.nan, "n_valid": int(valid_mask.sum())}
    m = clone(model)
    m.fit(X_data.loc[train_mask], y_data.loc[train_mask])
    p = np.clip(m.predict_proba(X_data.loc[valid_mask])[:, 1], 0.001, 0.999)
    return {
        "split": label,
        "brier": brier_score_loss(y_data.loc[valid_mask], p),
        "log_loss": log_loss(y_data.loc[valid_mask], p),
        "auc": roc_auc_score(y_data.loc[valid_mask], p),
        "n_valid": int(valid_mask.sum()),
    }


def stress_test_final_model(cols):
    X_data = train_labeled[cols].replace([np.inf, -np.inf], np.nan)
    model = build_models(X_data.shape[1])[FINAL_MODEL_NAME]
    cv_res, _ = evaluate_group_cv({FINAL_MODEL_NAME: model}, X_data, y, groups, n_splits=5)
    rows = [{
        "split": "group_kfold_aircraft",
        "brier": cv_res.iloc[0]["brier_oof"],
        "log_loss": cv_res.iloc[0]["log_loss_oof"],
        "auc": cv_res.iloc[0]["auc_oof"],
        "n_valid": len(y),
    }]

    periods = pd.PeriodIndex(train_labeled["year_month"], freq="M")
    month_num = periods.year * 12 + periods.month
    temporal_cut = np.quantile(month_num, 0.75)
    rows.append(evaluate_single_split(model, X_data, y, month_num <= temporal_cut, month_num > temporal_cut, "future_month_holdout"))

    delivery_year = pd.PeriodIndex(train_labeled["delivery_month_model"].astype(str), freq="M").year
    oldest_cut = np.quantile(delivery_year, 0.25)
    newest_cut = np.quantile(delivery_year, 0.75)
    rows.append(evaluate_single_split(model, X_data, y, delivery_year > oldest_cut, delivery_year <= oldest_cut, "oldest_delivery_cohort"))
    rows.append(evaluate_single_split(model, X_data, y, delivery_year < newest_cut, delivery_year >= newest_cut, "newest_delivery_cohort"))

    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE + 7)
    tr_idx, va_idx = next(gss.split(X_data, y, groups))
    train_mask = np.zeros(len(y), dtype=bool)
    valid_mask = np.zeros(len(y), dtype=bool)
    train_mask[tr_idx] = True
    valid_mask[va_idx] = True
    rows.append(evaluate_single_split(model, X_data, y, train_mask, valid_mask, "random_aircraft_20pct"))

    return pd.DataFrame(rows)


robustness_report = stress_test_final_model(feature_cols)
display(robustness_report)

robustness_summary = (
    pd.DataFrame([{
        "mean_brier": robustness_report["brier"].mean(),
        "worst_brier": robustness_report["brier"].max(),
        "std_brier": robustness_report["brier"].std(),
        "mean_auc": robustness_report["auc"].mean(),
        "n_features": len(feature_cols),
    }])
)
display(robustness_summary)

## Validation adversariale train/test


In [ ]:
def adversarial_train_test_validation(train_df, test_df, cols, sample_size=12000):
    rng = np.random.default_rng(RANDOM_STATE)
    n = min(sample_size, len(train_df), len(test_df))
    train_idx = rng.choice(len(train_df), size=n, replace=False)
    test_idx = rng.choice(len(test_df), size=n, replace=False)
    X_adv = pd.concat([
        train_df.iloc[train_idx][cols],
        test_df.iloc[test_idx][cols],
    ], ignore_index=True).replace([np.inf, -np.inf], np.nan)
    y_adv = pd.Series([0] * n + [1] * n)

    adv_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=400,
            min_samples_leaf=10,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    pred = np.zeros(len(y_adv))
    for tr_idx, va_idx in skf.split(X_adv, y_adv):
        m = clone(adv_model)
        m.fit(X_adv.iloc[tr_idx], y_adv.iloc[tr_idx])
        pred[va_idx] = m.predict_proba(X_adv.iloc[va_idx])[:, 1]

    adv_auc = roc_auc_score(y_adv, pred)
    adv_model.fit(X_adv, y_adv)
    importances = adv_model.named_steps["model"].feature_importances_
    adv_importance = (
        pd.DataFrame({"feature": cols, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(25)
    )
    return adv_auc, adv_importance


adv_auc, adv_importance = adversarial_train_test_validation(train_features_all, test_features_all, feature_cols)
print(f"Adversarial train/test AUC: {adv_auc:.3f}")
display(adv_importance)

if adv_auc > 0.80:
    print("Alerte: train/test sont faciles à distinguer. Ne fais pas confiance à un seul score local.")
elif adv_auc > 0.70:
    print("Attention: shift modéré train/test. Les stress tests et la calibration comptent beaucoup.")
else:
    print("Shift train/test raisonnable sur ce feature set.")

## Holdout 20 % des avions


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

holdout_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, holdout_idx = next(holdout_split.split(X, y, groups))

X_train_h, X_holdout = X.iloc[train_idx], X.iloc[holdout_idx]
y_train_h, y_holdout = y.iloc[train_idx], y.iloc[holdout_idx]
groups_train_h = groups.iloc[train_idx]
groups_holdout = groups.iloc[holdout_idx]

print("Avions train holdout :", groups_train_h.nunique())
print("Avions validation holdout :", groups_holdout.nunique())
print("Lignes train / validation :", len(X_train_h), len(X_holdout))

holdout_final_model = build_models(X_train_h.shape[1])[FINAL_MODEL_NAME]
holdout_calibrated_model = CalibratedClassifierCV(holdout_final_model, method="sigmoid", cv=5)
holdout_calibrated_model.fit(X_train_h, y_train_h)
holdout_pred = np.clip(holdout_calibrated_model.predict_proba(X_holdout)[:, 1], 0.001, 0.999)

final_holdout_score = pd.DataFrame([{
    "model": f"final_calibrated_{FINAL_MODEL_NAME}",
    "brier_holdout": brier_score_loss(y_holdout, holdout_pred),
    "log_loss_holdout": log_loss(y_holdout, holdout_pred),
    "auc_holdout": roc_auc_score(y_holdout, holdout_pred),
}])
display(final_holdout_score)

# Baselines simples pour vérifier qu'on bat la prédiction constante.
baseline_05 = np.full(len(y_holdout), 0.5)
baseline_rate = np.full(len(y_holdout), y_train_h.mean())
baseline_results = pd.DataFrame([
    {"model": "constant_0.5", "brier_holdout": brier_score_loss(y_holdout, baseline_05)},
    {"model": "train_positive_rate", "brier_holdout": brier_score_loss(y_holdout, baseline_rate)},
])
display(baseline_results)


def calibration_table(y_true, pred, n_bins=8):
    tmp = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(pred)})
    tmp["bin"] = pd.qcut(tmp["p"], q=min(n_bins, tmp["p"].nunique()), duplicates="drop")
    return (
        tmp.groupby("bin", observed=True)
        .agg(
            n=("y", "size"),
            pred_mean=("p", "mean"),
            observed_rate=("y", "mean"),
            abs_gap=("p", lambda s: np.nan),
        )
        .assign(abs_gap=lambda d: (d["pred_mean"] - d["observed_rate"]).abs())
        .reset_index()
    )


print("Calibration holdout - modèle final")
display(calibration_table(y_holdout, holdout_pred))

## Entraînement final + calibration


In [ ]:
final_base_model = build_models(X.shape[1])[FINAL_MODEL_NAME]
calibrated_model = CalibratedClassifierCV(final_base_model, method="sigmoid", cv=5)
calibrated_model.fit(X, y)


def predict_final_probability(X_new):
    X_new = X_new.replace([np.inf, -np.inf], np.nan)
    return np.clip(calibrated_model.predict_proba(X_new)[:, 1], 0.001, 0.999)

print("Modèle final unique entraîné sur toutes les références disponibles :", FINAL_MODEL_NAME)

## Importance des variables


In [ ]:
interpretation_pipeline = build_models(X.shape[1])[FINAL_MODEL_NAME]
interpretation_pipeline.fit(X, y)
selector = interpretation_pipeline.named_steps["select"]
importances = selector.scores_
feature_importance = (
    pd.DataFrame({"feature": feature_cols, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
display(feature_importance.head(25))

feature_importance["family"] = feature_importance["feature"].apply(feature_family)
family_importance = (
    feature_importance
    .groupby("family", as_index=False)
    .agg(
        total_importance=("importance", "sum"),
        mean_importance=("importance", "mean"),
        n_features=("feature", "count"),
        top_feature=("feature", "first"),
    )
    .sort_values("total_importance", ascending=False)
)
display(family_importance)

print("Lecture rapide :")
for _, row in family_importance.head(5).iterrows():
    print(f"- {row['family']}: top={row['top_feature']} | importance totale={row['total_importance']:.3f}")

if PLOTS_AVAILABLE:
    top = feature_importance.head(20).iloc[::-1]
    plt.figure(figsize=(9, 7))
    plt.barh(top["feature"], top["importance"])
    plt.title("Top variables - lecture modèle")
    plt.tight_layout()
    plt.show()

## Export soumission test


In [ ]:
test_scoring = test_features_all.copy()
test_scoring["id"] = test_scoring["aircraft_id"] + "_" + test_scoring["year_month"]

missing_features = sorted(set(feature_cols) - set(test_scoring.columns))
assert not missing_features, f"Features manquantes côté test : {missing_features[:10]}"

X_test_final = test_scoring[feature_cols].replace([np.inf, -np.inf], np.nan)
test_scoring["corrosion_risk"] = predict_final_probability(X_test_final)
test_scoring["corrosion_risk"] = test_scoring["corrosion_risk"].clip(0, 1)

submission = sample_sub[["id"]].merge(
    test_scoring[["id", "aircraft_id", "year_month", "corrosion_risk"]],
    on="id",
    how="left",
)

assert len(submission) == len(sample_sub)
assert submission["corrosion_risk"].notna().all()
assert submission["corrosion_risk"].between(0, 1).all()

display(submission.head())
display(submission["corrosion_risk"].describe())

In [ ]:
OUTPUT_PATH = Path("..") / "data" / "submissions" / "submission_final_model.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission[["id", "corrosion_risk"]].to_csv(OUTPUT_PATH, index=False)
print(f"Soumission écrite : {OUTPUT_PATH.resolve()}")
print(submission[["id", "corrosion_risk"]].head(10).to_string(index=False))


## Notes

- Prioriser les stress tests (cohorte, holdout temporel) avant le leaderboard public.
- Extensions possibles : distance mer, historique maintenance, modèle de survie.
